# 11.08 · Human-in-the-loop, guardrails, and fault tolerance

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 1.5–2 h |
| 📋 Prerequisites | Lessons 11.02, 11.04, 11.07; Lesson 04.04 (interrupts and human review). |
| 🔑 Keys | none for 🧪 cells · optional 💸 cell ≈ 5–8K tokens (off by default) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Pause** a deep agent before a risky tool call and resume it with approve, edit, or reject.
2. **Require** approval by path with `FilesystemPermission(mode="interrupt")`.
3. **Add** guardrails as middleware: PII redaction and per-tool call limits.
4. **Survive** rate limits and outages: retries with backoff, model fallback, graceful failure.

## 📚 Official reading (verified 2026-10-16, `deepagents` 0.7.26, `langchain` 1.4.6)

| Page | Read it for |
|---|---|
| [Human-in-the-loop (Deep Agents)](https://docs.langchain.com/oss/python/deepagents/human-in-the-loop) | `interrupt_on`, decisions, subagent interrupts |
| [Human-in-the-loop (LangChain)](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) | `HumanInTheLoopMiddleware`, the resume payload |
| [Guardrails](https://docs.langchain.com/oss/python/langchain/guardrails) | PII middleware, deterministic vs model-based guardrails |
| [Fault tolerance](https://docs.langchain.com/oss/python/deepagents/fault-tolerance) | Retries, fallbacks, limits, rate limiters |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "deepagents>=0.7.23,<0.8" "langchain-google-genai>=4.4.1,<5" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("deepagents", "langchain",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-hitl-guardrails'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Helpers: a refund tool worth guarding, and a model that fails on purpose.
from langchain.tools import tool
from langchain_core.exceptions import ModelRateLimitError
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from mastery.deep import build_deep_agent, run_stats
from mastery.testing import ScriptedChatModel


def calls(*items) -> AIMessage:
    return AIMessage("", tool_calls=[{"name": n, "args": a, "id": i} for n, a, i in items])


def user(text: str) -> dict:
    return {"messages": [{"role": "user", "content": text}]}


@tool
def issue_refund(invoice_id: str, amount: float) -> str:
    """Refund part or all of an invoice (EUR)."""
    return f"Refunded {amount:.2f} EUR on {invoice_id}."


class FlakyModel(ScriptedChatModel):
    """Raises a 429-style error for the first `fails` calls, then answers from its script."""
    fails: int = 2

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        if self.fails > 0:
            self.fails -= 1
            raise ModelRateLimitError("429 RESOURCE_EXHAUSTED (simulated)")
        return super()._generate(messages, stop, run_manager, **kwargs)

---
## Part 1 · Approve, edit, reject

In [ ]:
# 🧪 Human-in-the-loop: pause before issue_refund; a checkpointer is REQUIRED to pause and resume.
def refund_agent():
    script = ScriptedChatModel(responses=[calls(("issue_refund", {"invoice_id": "INV-2001", "amount": 500.0}, "r1")),
                                          AIMessage("Done.")])
    agent = build_deep_agent(script, tools=[issue_refund], checkpointer=InMemorySaver(), fallbacks=False,
                             interrupt_on={"issue_refund": {"allowed_decisions": ["approve", "edit", "reject"]}})
    return agent


agent = refund_agent()
config = {"configurable": {"thread_id": "refund-1"}}
paused = agent.invoke(user("Refund INV-2001."), config)
request = paused["__interrupt__"][0].value
print("paused on:", request["action_requests"][0]["name"], request["action_requests"][0]["args"])
print("allowed  :", request["review_configs"][0]["allowed_decisions"])

# The reviewer sees 500 EUR on a 50 EUR invoice and EDITS the call.
done = agent.invoke(Command(resume={"decisions": [{"type": "edit", "edited_action": {
    "name": "issue_refund", "args": {"invoice_id": "INV-2001", "amount": 50.0}}}]}), config)
print("\nexecuted :", [m.content.splitlines()[-1] for m in done["messages"] if m.type == "tool"])

**You should now see** the run paused on `issue_refund` with 500 EUR, and after the edit, "Refunded 50.00 EUR". The tool message also carries a notice that a human replaced the call, so the model does not retry its original one.

In [ ]:
# 🧪 Reject with a reason: the tool never runs and the model reads why.
agent = refund_agent()
config = {"configurable": {"thread_id": "refund-2"}}
agent.invoke(user("Refund INV-2001."), config)
done = agent.invoke(Command(resume={"decisions": [{"type": "reject", "message": "Over 200 EUR needs a manager."}]}), config)
print([m.content for m in done["messages"] if m.type == "tool"])

**What just happened?** The reject message became the tool result: the model knows *why* and can tell the customer or escalate. Always give a reason.

> ⚠️ **Production note: an interrupt is a stored, resumable run.** It needs a durable checkpointer (Postgres, a deployment's built-in one), not `InMemorySaver`. Resume with the same `thread_id`, show the reviewer the **arguments**, and log who decided what. Decisions must come in the same order as the action requests.

## Part 2 · Approval by path

In [ ]:
# 🧪 Interrupt by PATH instead of tool: writes under /reports/ need approval, scratch writes don't.
from deepagents import FilesystemPermission

script = ScriptedChatModel(responses=[calls(("write_file", {"file_path": "/notes.md", "content": "draft"}, "a"),
                                            ("write_file", {"file_path": "/reports/final.md", "content": "final"}, "b")),
                                      AIMessage("Saved.")])
agent = build_deep_agent(script, checkpointer=InMemorySaver(), fallbacks=False,
                         permissions=[FilesystemPermission(operations=["write"], paths=["/reports/**"], mode="interrupt")])
config = {"configurable": {"thread_id": "perm-1"}}
paused = agent.invoke(user("Write the notes and the report."), config)
print("paused on:", [(r["name"], r["args"]["file_path"]) for r in paused["__interrupt__"][0].value["action_requests"]])
done = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config)
print("files    :", sorted(done["files"]))

**You should now see** only `/reports/final.md` waiting for approval; `/notes.md` was written straight away. Path rules let you approve what leaves the scratchpad without slowing down every write.

## Part 3 · Guardrails

In [ ]:
# 🧪 Guardrails as middleware: redact emails before the model sees them; cap refunds to one per run.
from langchain.agents.middleware import PIIMiddleware, ToolCallLimitMiddleware

script = ScriptedChatModel(responses=[calls(("issue_refund", {"invoice_id": "INV-1", "amount": 10.0}, "a"),
                                            ("issue_refund", {"invoice_id": "INV-2", "amount": 10.0}, "b")),
                                      AIMessage("Done.")])
agent = build_deep_agent(script, tools=[issue_refund], fallbacks=False, middleware=[
    PIIMiddleware("email", strategy="redact", apply_to_input=True),
    ToolCallLimitMiddleware(tool_name="issue_refund", run_limit=1, exit_behavior="continue")])
out = agent.invoke(user("Refund INV-1 and INV-2, receipt to ana@example.com"))
print("model saw :", script.calls[0][-1].text)
for m in out["messages"]:
    if m.type == "tool":
        print("tool      :", m.content[:90])

**You should now see** `[REDACTED_EMAIL]` in what the model received, one refund executed, and the second blocked with a limit message. Both are deterministic: no model call, no tokens.

## Part 4 · Fault tolerance

In [ ]:
# 🧪 Fault tolerance: retry transient 429s, then fall back to another model, then fail gracefully.
from langchain.agents.middleware import ModelFallbackMiddleware, ModelRetryMiddleware

fast = dict(initial_delay=0.01, backoff_factor=1.0, jitter=False)   # real runs: keep the defaults (1 s, ×2, jitter)
rows = []

flaky = FlakyModel(responses=[AIMessage("Answer after two retries.")], fails=2)
out = build_deep_agent(flaky, fallbacks=False, middleware=[ModelRetryMiddleware(max_retries=3, **fast)]).invoke(user("Hi"))
rows.append({"case": "retry (2 failures)", "answer": out["messages"][-1].text})

dead = FlakyModel(responses=[AIMessage("never")], fails=99)
backup = ScriptedChatModel(responses=[AIMessage("Answer from the fallback model.")])
out = build_deep_agent(dead, fallbacks=False, middleware=[ModelFallbackMiddleware(backup)]).invoke(user("Hi"))
rows.append({"case": "fallback (quota gone)", "answer": out["messages"][-1].text})

dead = FlakyModel(responses=[AIMessage("never")], fails=99)
out = build_deep_agent(dead, fallbacks=False, middleware=[ModelRetryMiddleware(max_retries=1, **fast)]).invoke(user("Hi"))
rows.append({"case": "retries exhausted", "answer": out["messages"][-1].text[:80]})
table(rows, ["case", "answer"])

**What just happened?** Three layers: `ModelRetryMiddleware` absorbs short 429 bursts (backoff 1 s, ×2, with jitter by default); `ModelFallbackMiddleware` switches model when one is down or out of quota (`mastery.deep.build_deep_agent` adds it from `MASTERY_FALLBACK_MODELS`); and when everything fails, the run ends with a readable message instead of a traceback (`on_failure="continue"`).

> ⚠️ **Production note: retries on a daily quota just burn time.** A 429 for tokens per day won't clear in 60 s. In production: retry only transient errors, fall back on quota errors, alert when fallbacks fire, and keep the budgets from Lesson 11.02 so a loop can't spend the whole day's quota.

## Part 5 · 💸 You are the reviewer

> 💸 Calls your deep model (`MASTERY_DEEP_MODEL`, default Gemini free tier). Rate-limited to one call every 4 s and capped per run. Without the key it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE (opt-in): a real model proposes a refund; YOU decide. ≈ 5–8K tokens.
RUN_LIVE = False
if RUN_LIVE and HAVE_DEEP_KEY:
    live = build_deep_agent(tools=[issue_refund], checkpointer=InMemorySaver(), model_calls=6, tool_calls=4,
                            system_prompt="You are a billing agent. INV-2001 is a 50 EUR invoice.",
                            interrupt_on={"issue_refund": {"allowed_decisions": ["approve", "reject"]}})
    config = {"configurable": {"thread_id": "live-refund"}}
    paused = live.invoke(user("I was double charged on INV-2001, please refund it."), config)
    if "__interrupt__" in paused:
        print("proposed:", paused["__interrupt__"][0].value["action_requests"][0]["args"])
        decision = input("approve or reject? ").strip().lower()
        live_decision = {"type": "approve"} if decision == "approve" else {"type": "reject", "message": "Rejected by reviewer."}
        paused = live.invoke(Command(resume={"decisions": [live_decision]}), config)
    print("answer:", paused["messages"][-1].text[:300])
    print("stats :", run_stats(paused))
elif not RUN_LIVE:
    print("⏭️ Skipped: set RUN_LIVE = True to try it (≈ 5–8K tokens).")
else:
    print(f"⏭️ Skipped: no key for {DEEP_MODEL} (see the setup cell).")

**You should now see** (when enabled) the proposed arguments, your prompt, and the agent's reply after your decision.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| No `__interrupt__`, tool just runs | No checkpointer, or the tool name is misspelt in `interrupt_on` | Pass `checkpointer=`; match names |
| Resume starts a new run | Different `thread_id` | Reuse the paused config |
| `ValueError` on resume | Decision count ≠ pending action requests | One decision per action, in order |
| Retries hang for minutes | Daily quota, not a burst | Fallback instead; lower `max_retries` |
| Fallback never fires | Fallback has no key (skipped) | Check the setup cell's "fallbacks" line |

## 🏋️ Exercises

1. **(Easy)** In Part 1, approve instead of editing. What does the tool return?
2. **(Medium)** Add `"respond"` to the allowed decisions and resume with `{"type": "respond", "message": "Ask the customer for the bank details first."}`. What does the model receive?
3. **(Medium)** Add `PIIMiddleware("credit_card", strategy="mask", apply_to_input=True)` and send a test card number `4111 1111 1111 1111`.
4. **(Stretch)** Make the run end (`exit_behavior="end"`) when a second refund is attempted, and compare the final message with Part 3.

<details>
<summary><b>Show solutions</b></summary>

**1.** "Refunded 500.00 EUR on INV-2001.": approve runs the original arguments, which is exactly why the reviewer must read them.

**2.** The tool doesn't run; your text comes back as the tool result, so the model can follow the instruction.

**3.** The model sees `card **** **** **** 1111`: masked to the last four digits.

**4.** `ToolCallLimitMiddleware(tool_name="issue_refund", run_limit=1, exit_behavior="end")`: the run stops and the last message is the limit notice ("'issue_refund' tool call limit reached …") instead of the model's own answer.
</details>

## 🔎 Docs drill (5 min)

A subagent calls a tool that is in the main agent's interrupt_on. Does the run still pause, and where do you configure different rules for that subagent?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Human-in-the-loop (Deep Agents)](https://docs.langchain.com/oss/python/deepagents/human-in-the-loop).
2. Read the `interrupt_on` section of `??create_deep_agent` (from `deepagents`).
3. Declarative subagents (dicts) inherit the top-level `interrupt_on`; an `interrupt_on` in the subagent dict overrides it. A `CompiledSubAgent` does **not** inherit it: configure approval inside its runnable.

</details>

## ✅ Checkpoint

- [ ] I can pause, edit, approve and reject a tool call, and I know why a checkpointer is required.
- [ ] I can require approval by path.
- [ ] I can add PII and call-limit guardrails without extra model calls.
- [ ] I can explain retry vs fallback vs graceful failure, and when each applies.
- [ ] Restart & Run All passes without keys; outputs cleared before saving.

## 🔎 Research checkpoint

On [Fault tolerance](https://docs.langchain.com/oss/python/deepagents/fault-tolerance), check the current defaults of `ModelRetryMiddleware` and whether a new error type is retried by default. Log differences with versions in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- `interrupt_on` (by tool) and `mode="interrupt"` (by path) pause runs for a human; resume with `Command(resume=...)`.
- Deterministic guardrails (PII, call limits) are cheap and belong in every agent.
- Retry bursts, fall back on outages, end gracefully, and keep budgets.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 11-09-code-execution-sandboxes.ipynb** → [`11-09-code-execution-sandboxes.ipynb`](11-09-code-execution-sandboxes.ipynb).

[Module 11 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)